_Date: 2026-08-14_


# Amplitude Rabi Chevron

2D sweep of drive **frequency** x drive **amplitude**, using the `contrib` experiment
`amplitude_rabi_chevron` from `laboneq_applications`. Unlike the 1D `amplitude_rabi`
experiment, this workflow is diagnostic only: it has no `evaluate_experiment`/`update_qpu`
step, so it never writes anything back to the QPU -- it just runs and (optionally) plots
the 2D chevron pattern.


In [ ]:
import numpy as np
from laboneq.simple import *

from laboneq_applications.qpu_types.tunable_transmon import (
    TunableTransmonQubit,
    TunableTransmonOperations,
)
from laboneq_applications.contrib.experiments import amplitude_rabi_chevron


## 1. Device Setup & QPU -- SHFQC (SG drive / QA readout)

Single qubit `q0`, driven on an SG channel, read out on a QA channel of the same SHFQC.
Built via the programmatic `DeviceSetup` API (`add_instruments`/`add_connections`),
matching ZI's official `getting_started_shfqc_plus` example -- `DeviceSetup.from_descriptor()`'s
YAML grammar only supports separate `SHFSG`/`SHFQA` entries, not the combined `SHFQC`
instrument type, and raises `AssertionError: Invalid device type` from the emulator if
you try `SHFQC:` as a descriptor key.

**Fill in before running on real hardware:**
- `address` -- your SHFQC's device serial (e.g. `dev12345`, check the LabOne Web UI or the
  label on the front panel).
- `device_options` -- your SHFQC's installed options string (LabOne Web UI -> Device tab,
  or `*OPT?`). Must contain a channel-count token (`QC2CH`/`QC4CH`/`QC6CH`) matching your
  instrument's SG channel count; drop `/PLUS/16W` if you don't have those options.
- `host` -- the LabOne dataserver host (`"localhost"` if running on the same PC as the
  dataserver).
- Every `qubit.parameters.*` value below -- these are placeholders. Use your qubit's
  already-calibrated values (e.g. from your 1D `amplitude_rabi` run) rather than these.

`do_emulation=True` is kept as the default below so nothing is sent to the instrument
until you've checked the values above -- flip to `False` once confirmed.


In [ ]:
device_setup = DeviceSetup(uid="shfqc_setup")
device_setup.add_dataserver(host="10.42.193.4", port="8004")

shfqc = SHFQC(
    uid="device_shfqc",
    address="dev12073",
    interface="1GbE",
    device_options="SHFQC/LRT/PLUS/QC6CH/RTR",
    reference_clock_source="internal",
)
device_setup.add_instruments(shfqc)

device_setup.add_connections(
    "device_shfqc",
    create_connection(to_signal="q0/drive", ports="SGCHANNELS/0/OUTPUT", type="iq"),
    create_connection(to_signal="q0/measure", ports="QACHANNELS/0/OUTPUT", type="iq"),
    create_connection(to_signal="q0/acquire", ports="QACHANNELS/0/INPUT", type="acquire"),
)

q_uid = "q0"
qubits = TunableTransmonQubit.from_device_setup(device_setup)
qubit = next(q for q in qubits if q.uid == q_uid)

# TODO: replace every value below with your qubit's real calibrated parameters
qubit.parameters.resonance_frequency_ge = 5.0e9        # qubit ge transition frequency
qubit.parameters.drive_lo_frequency = 4.8e9            # SG channel LO frequency
qubit.parameters.readout_resonator_frequency = 7.0e9   # readout resonator frequency
qubit.parameters.readout_lo_frequency = 6.8e9          # QA channel LO frequency
qubit.parameters.ge_drive_amplitude_pi = 0.8           # calibrated pi-pulse amplitude
qubit.parameters.ge_drive_amplitude_pi2 = 0.4          # calibrated pi/2-pulse amplitude

qpu = QPU(quantum_elements=qubits, quantum_operations=TunableTransmonOperations())

session = Session(device_setup)
session.connect(do_emulation=False)   # set False once the address/options/parameters above are confirmed


## 2. Frequency x Amplitude Sweep

- **Frequency**: +/-20 MHz around the qubit's calibrated `resonance_frequency_ge`, 41 points.
- **Amplitude**: 0 to 1 (full drive-amplitude scale), 21 points.

Adjust `N_FREQ`/`N_AMP` or the frequency span to trade resolution against run time
(total real-time shots ~= `N_FREQ * N_AMP * count`).


In [ ]:
N_FREQ = 41
N_AMP = 21
FREQ_SPAN = 20e6   # +/- around resonance_frequency_ge

f_center = qubit.parameters.resonance_frequency_ge
frequencies = f_center + np.linspace(-FREQ_SPAN, FREQ_SPAN, N_FREQ)
amplitudes = np.linspace(0, 1, N_AMP)

print(f"Sweeping {N_FREQ} frequencies around {f_center/1e9:.4f} GHz, {N_AMP} amplitudes 0-1")


## 2a. Add a Marker to the Drive Pulse

`amplitude_rabi_chevron`'s `create_experiment` plays the drive pulse via `qop.x180(q,
amplitude=amplitude)`, which delegates to `rx()` -- there's no `marker=` argument exposed
on `x180`/`rx`, so a marker can't be passed in through the workflow call itself.

Instead, `rx` is overridden on `qpu.quantum_operations` (the same override mechanism
`laboneq_applications` itself uses to register operations) so it plays with
`marker={"marker1": {"enable": True}}` every time -- `x180` (and therefore the chevron's
drive pulse) calls `self.rx(...)` internally, so this applies automatically without
touching the installed library. `marker1` is enabled for the duration of the drive pulse
on every sweep point, so a scope can trigger on it in sync with each drive pulse, same as
the chirp pulse notebook.


In [ ]:
@qpu.quantum_operations.register
def rx(self, q, angle, transition=None, amplitude=None, phase=0.0,
       increment_oscillator_phase=None, length=None, pulse=None) -> None:
    drive_line, params = q.transition_parameters(transition)
    if transition == "ef":
        dsl.active_section().on_system_grid = True
    if amplitude is None:
        amplitude = (angle / np.pi) * params["amplitude_pi"]
    if length is None:
        length = params["length"]
    rx_pulse = dsl.create_pulse(params["pulse"], pulse, name="rx_pulse")
    dsl.play(
        q.signals[drive_line],
        amplitude=amplitude,
        phase=phase,
        increment_oscillator_phase=increment_oscillator_phase,
        length=length,
        pulse=rx_pulse,
        marker={"marker1": {"enable": True}},
    )


## 3. Run the Chevron Experiment


In [ ]:
options = amplitude_rabi_chevron.experiment_workflow.options()
options.count(1024)
options.use_cal_traces(True)

chevron_result = amplitude_rabi_chevron.experiment_workflow(
    session=session,
    qpu=qpu,
    qubits=q_uid,
    frequencies=frequencies,
    amplitudes=amplitudes,
    options=options,
).run()

chevron_result.tasks


## 3a. Pulse Sheet Viewer

Renders the compiled experiment's pulse sequence (drive/measure/reset, per sweep step)
as an interactive HTML file, then force-opens it in your **system default browser** --
the file is a self-contained ~1.4 MB JS bundle, not a CDN-loaded chart, so VS Code's
notebook link/preview can't execute it properly (shows raw text instead of the chart).

The full 2D sweep has `N_FREQ * N_AMP` real-time steps -- with the current 41x21 settings
that's 861, which can make the pulse sheet large/slow to render and may hit
`max_events_to_publish` truncation (increased below, but raise further if you still see a
truncation warning). For a quick structural sanity check instead of the full sweep, drop
`N_FREQ`/`N_AMP` to e.g. 2x2 before running Section 3.


In [ ]:
import glob
import os
import webbrowser

compiled_exp = chevron_result.tasks["compile_experiment"].output
show_pulse_sheet("chevron_pulse_sheet", compiled_exp, max_events_to_publish=5000)

# Force-open in the system browser -- VS Code's notebook viewer can't run the pulse
# sheet's embedded JS bundle, so the IPython link/preview shows raw text instead.
latest_pulse_sheet = max(glob.glob("chevron_pulse_sheet_*.html"), key=os.path.getmtime)
webbrowser.open(f"file://{os.path.abspath(latest_pulse_sheet)}")
print(f"Opened {latest_pulse_sheet} in your default browser")


## 4. Inspect Results

`analysis_workflow` (run automatically above via `options.do_analysis`, default `True`)
already plots the 2D chevron pattern. This cell pulls the raw acquired data directly, in
case you want a custom plot instead.


In [ ]:
acquired_data = chevron_result.tasks["run_experiment"].output
raw = acquired_data[q_uid].result
print("Raw result array shape (amplitude x frequency):", np.shape(raw))
